# 06 — Export to AutoGen

Take a generated topology and export it as an AutoGen team config.

**Prerequisites:**
- Python 3.10+ with numpy
- Running from inside the `flybrain-connectome` checkout
- Completed [01_generate_visualize.ipynb](01_generate_visualize.ipynb)

## Cell 1 — Load a previously generated topology

In [ ]:
import sys, os, json
from pathlib import Path

HERE = Path(__file__).resolve().parents[1]
TOOLKIT = HERE / "flybrain-connectome-mcp"
for p in [TOOLKIT, TOOLKIT / "tools"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Load the topology
org_dir = TOOLKIT / "generated_organisms"
json_path = org_dir / "demo_predictive_coding_v2_m5_a8.json"
if not json_path.exists():
    print("[warn] Run 'flybrain demo' first to generate a topology")
    import subprocess
    subprocess.run([sys.executable, str(HERE / "flybrain_cli.py"), "demo", "--no-open"],
                   cwd=str(HERE), check=True)

with open(json_path, "r", encoding="utf-8") as f:
    topo = json.load(f)

nodes = topo.get("organism", {}).get("nodes", [])
edges = topo.get("organism", {}).get("edges", [])
print(f"[loaded] {len(nodes)} nodes, {len(edges)} edges")

## Cell 2 — Export to AutoGen format

In [ ]:
# Use the export command
import subprocess

result = subprocess.run(
    [sys.executable, str(HERE / "flybrain_cli.py"), "export", str(json_path),
     "--format", "autogen", "--output", str(json_path.parent / "autogen_config.json")],
    capture_output=True, text=True, cwd=str(HERE)
)
print(result.stdout)
if result.stderr:
    print("[stderr]", result.stderr)

## Cell 3 — Inspect the AutoGen config

In [ ]:
with open(json_path.parent / "autogen_config.json", "r", encoding="utf-8") as f:
    autogen = json.load(f)

print("AutoGen config structure:")
print(f"  blueprint : {autogen['blueprint']}")
print(f"  format    : {autogen['format']}")
print(f"  team type : {autogen['team']['type']}")
print(f"  agents    : {len(autogen['team']['agents'])}")
print(f"  edges     : {len(autogen['edges'])}")
print()

# Show first 3 agents
print("First 3 agents:")
for a in autogen['team']['agents'][:3]:
    print(f"  name: {a['name']}")
    print(f"    role: {a['role']}")
    print(f"    system: {a['system_message'][:80]}...")
    print(f"    module: {a['module']}")

## Cell 4 — Use with AutoGen (pseudocode)

In [ ]:
# Pseudocode for AutoGen integration (requires pyautogen installed)
# The config maps directly to AutoGen Agent + GroupChat.

# from autogen import GroupChat, GroupChatManager
# from autogen.agentchat.contrib.capabilities.agent_builder import AssistantBuilder

# # Create agents
# agents = []
# for agent_def in autogen['team']['agents']:
#     builder = AssistantBuilder(
#         model_config={
#             "model": agent_def['llm_config']['model'],
#             "temperature": agent_def['llm_config']['temperature'],
#         },
#         system_message=agent_def['system_message'],
#     )
#     agent = builder.create_initial_assistant()
#     agents.append(agent)

# # Create group chat
# group_chat = GroupChat(
#     agents=agents,
#     message_kickoff=autogen['team']['group_chat']['message_kickoff'],
#     speaker_selection_method=autogen['team']['group_chat']['speaker_selection_method'],
#     max_round=autogen['team']['group_chat']['max_round'],
# )

# # Create manager
# manager = GroupChatManager(groupchat=group_chat)

# # Run
# assistant = agents[0]
# assistant.receive(autogen['team']['group_chat']['message_kickoff'])
# assistant.initiate_chat(manager)

print("AutoGen integration ready!")
print(f"  Agents to create: {len(autogen['team']['agents'])}")
print(f"  Edges to wire   : {len(autogen['edges'])}")

## Key insight

AutoGen is great for multi-agent conversations with dynamic speaker
selection. The flybrain topology gives you a **biologically-grounded**
conversation:

- Each agent = a neural role (sensory, motor, associative, etc.)
- Each edge = a real synapse class (excitatory, inhibitory, modulatory)
- Each module = a brain region (cerebrum, cerebellum, etc.)

The group chat dynamics are guided by real neural connectivity patterns.
This is not just a conversation — it's a **provably-traceable** agent
conversation that can be audited with `flybrain audit`.

## Summary: Framework Export Comparison

| Framework | Best For | Graph Type | Config Format |
|---|---|---|---|
| **LangGraph** | Stateful workflows | StateGraph | YAML/JSON |
| **CrewAI** | Role-playing crews | Sequential/Hierarchical | YAML/JSON |
| **AutoGen** | Dynamic conversations | GroupChat | JSON |

All three configs are generated from the same flybrain topology,
ensuring **consistency across frameworks** while allowing each
framework to use its native paradigm.

## Next

- Back to [03_provenance_audit.ipynb](03_provenance_audit.ipynb) — audit your exported config
- Or try a different blueprint with `flybrain demo --blueprint <name>`